# Assignment 4 — Advanced Spatial Operations

Implements defect detection, block analysis, contrast enhancement, and gradients.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [ ]:
def defects_cv(image):
    threshold = image.min() + (image.max() - image.min()) / 3
    mask = cv2.inRange(image, int(image.min()), int(threshold))
    enhanced = image.copy(); enhanced[mask > 0] = 255
    block_sum = cv2.boxFilter((mask > 0).astype(np.uint8), -1, (3, 3), normalize=False)
    return mask, enhanced, block_sum

def contrast_and_edges_cv(image):
    enhanced = cv2.equalizeHist(image)
    edges = cv2.Sobel(enhanced, cv2.CV_32F, 1, 1, ksize=3)
    return enhanced, np.abs(edges).astype(np.uint8)


## Style 2 — DSA/manual implementation

In [ ]:
def defects_manual(image):
    threshold = int(image.min() + (image.max() - image.min()) / 3)
    mask = (image <= threshold).astype(np.uint8)
    block_sum = np.zeros_like(image, np.int32)
    for r in range(image.shape[0] - 2):
        for c in range(image.shape[1] - 2):
            block_sum[r, c] = int(mask[r:r + 3, c:c + 3].sum())
    enhanced = image.copy(); enhanced[mask == 1] = 255
    return mask, enhanced, block_sum

def gradient_manual(image):
    gx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    gy = gx.T; padded = np.pad(image.astype(float), 1, mode="edge")
    result = np.zeros_like(image, float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            w = padded[r:r + 3, c:c + 3]
            result[r, c] = min(255, np.hypot((w * gx).sum(), (w * gy).sum()))
    return result.astype(np.uint8)


## Demonstration

In [ ]:
image = cv2.imread(str(LAB_DIR / "night_road.jpg"), cv2.IMREAD_GRAYSCALE)
if image is None: image = np.arange(64, dtype=np.uint8).reshape(8, 8) * 4
mask, enhanced, _ = defects_cv(image); manual_mask, manual_enhanced, _ = defects_manual(image)
print("Defect-mask equality:", np.array_equal(mask > 0, manual_mask > 0))
show_images([image, enhanced, manual_enhanced], ["Original", "cv2 threshold result", "Manual result"])
